AWS IAM — Identity and Access Management


⸻

1. What is AWS IAM?

IAM stands for Identity and Access Management.

AWS IAM is a service used to control:

* Who can access AWS
* What they can access
* What actions they can perform
* Under what conditions they can perform those actions

The easiest way to remember IAM is:

WHO
 |
 v
User / Role
 |
 v
WANTS TO DO WHAT?
 |
 v
Action
 |
 v
ON WHICH RESOURCE?
 |
 v
Resource
 |
 v
IS IT ALLOWED?
 |
 v
IAM Policy Evaluation

Example:

User: Anuj
Action:
s3:GetObject
Resource:
company-data bucket
Result:
Allowed / Denied

IAM is a global AWS service. IAM users, groups, roles, and policies are not created separately for each AWS Region.

⸻

2. Authentication vs Authorization

These are two fundamental IAM concepts.

Authentication

Authentication answers:

Who are you?

Examples:

Username + Password
Access Key + Secret Access Key
MFA
Temporary security credentials

Example:

Anuj
  |
  | Username + Password
  v
AWS
AWS verifies:
"Is this really Anuj?"

That is Authentication.

⸻

Authorization

Authorization answers:

What are you allowed to do?

After AWS knows who you are, AWS checks your permissions.

Example:

Anuj logged in successfully
        |
        v
Can Anuj read S3?
        |
       YES

But:

Can Anuj terminate EC2?
        |
       NO

That is Authorization.

⸻

Authentication vs Authorization

Authentication	Authorization
Who are you?	What can you do?
Verifies identity	Checks permissions
Happens first	Happens after authentication
Password, MFA, keys	IAM policies
Example: Login	Example: Access S3

Interview Answer

Q: What is the difference between Authentication and Authorization?

Authentication verifies the identity of a user or application, while authorization determines what actions that authenticated identity is allowed to perform on AWS resources.

⸻

3. Main Components of IAM

The major IAM components are:

IAM
 |
 +---- Users
 |
 +---- Groups
 |
 +---- Policies
 |
 +---- Roles
 |
 +---- Temporary Credentials / STS

Simple definition:

IAM User
= Individual identity
IAM Group
= Collection of users
IAM Policy
= Permission document
IAM Role
= Assumable identity
STS
= Temporary credentials

⸻

4. IAM User

An IAM User represents an identity that requires AWS access.

Example:

Company Employees
Anuj
Rahul
Priya
John

IAM users:

anuj
rahul
priya
john

Each user can have different permissions.

Example:

Anuj
 |
 +-- S3 Read/Write
 +-- Glue Access
 +-- Athena Access
Rahul
 |
 +-- EC2 Read
 +-- CloudWatch Read

⸻

5. How IAM Users Access AWS

There are two broad ways.

AWS Management Console

Usually through:

Username
+
Password
+
MFA

Example:

Browser
   |
   v
AWS Login
   |
   v
IAM User

⸻

Programmatic Access

Applications or tools may access AWS using credentials.

Examples:

AWS CLI
Python boto3
Terraform
AWS SDK
Scripts

Long-term IAM user credentials can include:

Access Key ID
Secret Access Key

For AWS workloads, using IAM Roles with temporary credentials is generally preferred over storing permanent access keys.

⸻

6. IAM Group

An IAM Group is:

A collection of IAM users that require similar permissions.

Example:

DataEngineers
 |
 +-- Anuj
 +-- Rahul
 +-- Priya
 +-- Amit

Instead of attaching policies separately:

Anuj  -> S3 + Glue + Athena
Rahul -> S3 + Glue + Athena
Priya -> S3 + Glue + Athena

Create:

DataEngineers Group
        |
        +-- S3
        +-- Glue
        +-- Athena

Then add users:

DataEngineers
     |
  --------------
  |     |      |
Anuj  Rahul  Priya

All users receive the group’s permissions.

⸻

7. Why Use IAM Groups?

Suppose you have:

100 Data Engineers

Without groups:

Attach policies individually to 100 users

With groups:

Create one:
DataEngineers Group
Attach policies once
Add 100 users

Benefits:

Centralized management
Easy onboarding
Easy offboarding
Consistent permissions
Reduced administrative effort

⸻

8. Important IAM Group Rules

IAM Groups contain:

IAM Users

They do not contain:

IAM Roles
Other IAM Groups
AWS Services

A user can belong to multiple groups.

Example:

                Anuj
               /    \
              /      \
             v        v
DataEngineers     ProductionSupport

Anuj receives applicable permissions from both groups.

Groups themselves do not have:

Password
Access Key
Login credentials

The user authenticates, not the group.

⸻

9. IAM Policy

An IAM Policy is:

A JSON document that defines permissions.

A policy specifies:

Allow or Deny
Which Action?
Which Resource?
Under what Condition?

Basic example:

{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": "s3:GetObject",
      "Resource": "arn:aws:s3:::company-data/*"
    }
  ]
}

Meaning:

Allow
   |
   v
s3:GetObject
   |
   v
company-data bucket objects

⸻

10. IAM Policy Structure

Important components:

Version
Statement
Sid
Effect
Action
Resource
Condition
Principal

⸻

11. Version

Example:

"Version": "2012-10-17"

This is the IAM policy language version.

It is not the policy creation date.

⸻

12. Statement

Statement contains one or more permission rules.

Example:

"Statement": [
    {
        "Effect": "Allow",
        "Action": "s3:GetObject",
        "Resource": "arn:aws:s3:::company-data/*"
    }
]

A policy can contain multiple statements.

Example:

{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": "s3:GetObject",
      "Resource": "arn:aws:s3:::company-data/*"
    },
    {
      "Effect": "Allow",
      "Action": "glue:StartJobRun",
      "Resource": "*"
    }
  ]
}

The identity can now:

Read S3 objects
Start Glue jobs

⸻

13. Sid

Sid means:

Statement ID

Example:

"Sid": "AllowReadFromS3"

It makes policies easier for humans to understand.

Example:

{
  "Sid": "AllowReadFromRawBucket",
  "Effect": "Allow",
  "Action": "s3:GetObject",
  "Resource": "arn:aws:s3:::raw-data/*"
}

⸻

14. Effect

Effect determines whether permission is:

Allow
or
Deny

Example:

"Effect": "Allow"

or:

"Effect": "Deny"

⸻

15. Action

Action specifies the AWS API operation.

S3 examples:

s3:GetObject
s3:PutObject
s3:DeleteObject
s3:ListBucket

EC2 examples:

ec2:StartInstances
ec2:StopInstances
ec2:DescribeInstances

Glue examples:

glue:StartJobRun
glue:GetJob
glue:GetJobRun
glue:CreateJob

Lambda example:

lambda:InvokeFunction

⸻

16. Multiple Actions

Example:

"Action": [
    "s3:GetObject",
    "s3:PutObject"
]

This allows:

Read objects
Write objects

But does not automatically allow:

Delete objects

⸻

17. Wildcards

Example:

"Action": "s3:*"

Means:

All S3 actions

This could include:

GetObject
PutObject
DeleteObject
CreateBucket
DeleteBucket
ListBucket
...

Broad wildcards should be avoided where unnecessary.

Prefer:

"Action": [
    "s3:GetObject",
    "s3:PutObject"
]

when those are the only required actions.

⸻

18. Resource

Resource specifies:

Which AWS resource does this permission apply to?

Example:

"Resource": "arn:aws:s3:::company-data/*"

⸻

19. ARN

ARN stands for:

Amazon Resource Name

Think of it as the unique address of an AWS resource.

Example S3 bucket:

arn:aws:s3:::company-data

Objects:

arn:aws:s3:::company-data/*

⸻

20. Very Important S3 Resource Concept

Suppose your bucket is:

company-data

Bucket ARN:

arn:aws:s3:::company-data

Object ARN:

arn:aws:s3:::company-data/*

s3:ListBucket applies to the bucket.

Example:

{
  "Effect": "Allow",
  "Action": "s3:ListBucket",
  "Resource": "arn:aws:s3:::company-data"
}

s3:GetObject applies to objects.

{
  "Effect": "Allow",
  "Action": "s3:GetObject",
  "Resource": "arn:aws:s3:::company-data/*"
}

This distinction causes many real-world AccessDenied problems.

⸻

21. Complete S3 Example

Requirement:

List bucket
Read files
Write files

Policy:

{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": "s3:ListBucket",
      "Resource": "arn:aws:s3:::sales-data-prod"
    },
    {
      "Effect": "Allow",
      "Action": [
        "s3:GetObject",
        "s3:PutObject"
      ],
      "Resource": "arn:aws:s3:::sales-data-prod/*"
    }
  ]
}

⸻

22. Resource Wildcard

Example:

"Resource": "*"

Means:

All applicable resources

Some AWS API actions do not support fine-grained resource restrictions and therefore use:

"*"

However, when resource-level permissions are supported, restrict the resource whenever possible.

⸻

23. Condition

Condition allows permissions only when specific requirements are met.

Concept:

Allow Action
      |
      v
ONLY IF condition is true

Examples:

Access only from specific IP
Require MFA
Access only through specific VPC endpoint
Restrict by tags
Restrict by organization
Restrict by time

Example:

"Condition": {
    "IpAddress": {
        "aws:SourceIp": "203.0.113.0/24"
    }
}

Conceptually:

Company Network
       |
       v
Access Allowed
Other Network
       |
       v
Access Not Allowed

⸻

24. Principle of Least Privilege

One of the most important security principles.

Definition:

Give an identity only the minimum permissions required to perform its job.

Bad:

{
  "Effect": "Allow",
  "Action": "*",
  "Resource": "*"
}

This effectively grants extremely broad permissions.

Better:

{
  "Effect": "Allow",
  "Action": [
    "s3:GetObject",
    "s3:PutObject"
  ],
  "Resource": "arn:aws:s3:::company-data/*"
}

⸻

25. IAM Policy Types

Important policy types:

AWS Managed Policy
Customer Managed Policy
Inline Policy

⸻

26. AWS Managed Policy

Created and maintained by AWS.

Examples include managed policies for common AWS service access patterns.

Advantages:

Easy to use
Quick setup
AWS maintains policy

Limitation:

May grant more permissions than your exact application requires

⸻

27. Customer Managed Policy

Created and managed by your organization.

Example:

CompanyDataEngineerPolicy

Requirement:

Read company-raw bucket
Write company-processed bucket

Example:

{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": "s3:GetObject",
      "Resource": "arn:aws:s3:::company-raw/*"
    },
    {
      "Effect": "Allow",
      "Action": "s3:PutObject",
      "Resource": "arn:aws:s3:::company-processed/*"
    }
  ]
}

Customer managed policies are reusable.

                 CompanyPolicy
                       |
              -------------------
              |        |        |
            Role A   Role B   Group A

⸻

28. Inline Policy

Inline policies are embedded directly into one:

User
Group
Role

Example:

FinanceGlueRole
       |
       v
Inline Policy

They are useful when a permission is tightly coupled to a specific identity.

⸻

29. Managed vs Inline Policies

Managed Policy	Inline Policy
Separate IAM resource	Embedded directly
Reusable	Not independently reusable
Can attach to multiple identities	Tied to one identity
Easier centralized management	Useful for unique permissions

⸻

30. IAM Role

An IAM Role is:

An AWS identity that can be assumed to obtain permissions temporarily.

Think:

IAM User
=
Who I permanently am
IAM Role
=
What responsibility I temporarily perform

Example:

Anuj
 |
 | Assume
 v
ProductionSupportRole

⸻

31. Why IAM Roles?

Suppose EC2 needs S3 access.

Bad design:

EC2
 |
 | Hardcoded access key
 | Hardcoded secret key
 v
S3

Problems:

Credential leakage
Manual rotation
Security risk
Credentials could be committed to Git

Better:

EC2
 |
 v
IAM Role
 |
 v
Temporary Credentials
 |
 v
S3

⸻

32. EC2 Role Example

Application running on EC2:

import boto3
s3 = boto3.client("s3")

No hardcoded AWS credentials are required when the instance is correctly configured with an IAM role.

Architecture:

EC2
 |
 v
EC2DataRole
 |
 v
Temporary Credentials
 |
 v
S3

⸻

33. Glue IAM Role Example

Data engineering pipeline:

S3 Raw
   |
   v
Glue Job
   |
   v
S3 Processed

Glue might require:

s3:GetObject
s3:PutObject
s3:ListBucket
Glue Catalog access
CloudWatch Logs access

Create:

GlueETLRole

Architecture:

                 GlueETLRole
                     |
        -----------------------------
        |             |             |
        v             v             v
       S3       Glue Catalog   CloudWatch
        ^
        |
      Glue

⸻

34. Lambda IAM Role Example

Architecture:

S3
 |
 | Object Created
 v
Lambda
 |
 +------> DynamoDB
 |
 +------> CloudWatch

Lambda Role permissions could include:

s3:GetObject
dynamodb:PutItem
logs:CreateLogStream
logs:PutLogEvents

⸻

35. Two Important Parts of IAM Role

Every role conceptually involves:

IAM Role
 |
 +---- Trust Policy
 |
 +---- Permission Policy

Remember:

Trust Policy
=
WHO can assume the role?
Permission Policy
=
WHAT can the role do?

⸻

36. Trust Policy

Example for EC2:

{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Principal": {
        "Service": "ec2.amazonaws.com"
      },
      "Action": "sts:AssumeRole"
    }
  ]
}

Meaning:

Allow EC2 service
to assume this role.

⸻

37. Glue Trust Policy

{
  "Effect": "Allow",
  "Principal": {
    "Service": "glue.amazonaws.com"
  },
  "Action": "sts:AssumeRole"
}

Meaning:

Glue can assume this role.

⸻

38. Lambda Trust Policy

{
  "Effect": "Allow",
  "Principal": {
    "Service": "lambda.amazonaws.com"
  },
  "Action": "sts:AssumeRole"
}

⸻

39. Permission Policy

Permission policy determines what the role can do after being assumed.

Example:

{
  "Effect": "Allow",
  "Action": [
    "s3:GetObject",
    "s3:PutObject"
  ],
  "Resource": "arn:aws:s3:::company-data/*"
}

Therefore:

Trust Policy
      |
      v
Who can become this role?
Permission Policy
      |
      v
What can that role do?

⸻

40. STS — Security Token Service

STS stands for:

AWS Security Token Service

STS provides:

Temporary Security Credentials

When a principal assumes a role:

User / Service
      |
      v
STS AssumeRole
      |
      v
IAM Role
      |
      v
Temporary Credentials

⸻

41. Temporary Credentials

Temporary AWS credentials generally contain:

Access Key ID
Secret Access Key
Session Token
Expiration

Important difference:

IAM User Access Key
        |
        v
Long-term credential
Role / STS credentials
        |
        v
Temporary credentials

⸻

42. AssumeRole

sts:AssumeRole allows a principal to temporarily operate using an IAM role.

Example:

Anuj
 |
 | sts:AssumeRole
 v
ProductionReadRole
 |
 v
Production S3

⸻

43. Cross-Account Access

Suppose:

Account A
=
Development
Account B
=
Production

A user from development needs to read production data.

Instead of creating another IAM user in Account B:

Account B
Create:
ProductionReadRole

Trust:

Allow Account A principal
to assume ProductionReadRole

Permission:

Allow read from Production S3

Flow:

Account A User
      |
      | AssumeRole
      v
ProductionReadRole
      |
      v
Production S3

This is a common enterprise design.

⸻

44. IAM Policy Evaluation Logic

This is one of the most important interview topics.

Think:

Request comes to AWS
        |
        v
Is there an Explicit DENY?
        |
    YES +----> DENY
        |
       NO
        |
        v
Is there applicable ALLOW?
        |
      YES
        |
        v
      ALLOW
No Allow
        |
        v
Implicit DENY

⸻

45. Implicit Deny

By default:

Everything is denied

until access is allowed.

Example:

If your policy allows:

s3:GetObject

but says nothing about:

s3:DeleteObject

DeleteObject is:

Implicitly Denied

⸻

46. Explicit Deny

Example:

Policy 1:

{
  "Effect": "Allow",
  "Action": "s3:*",
  "Resource": "*"
}

Policy 2:

{
  "Effect": "Deny",
  "Action": "s3:DeleteObject",
  "Resource": "*"
}

Result:

GetObject       
PutObject       
ListBucket      
DeleteObject    

Remember:

EXPLICIT DENY
      >
ALLOW

⸻

47. Identity-Based Policy

Attached to:

IAM User
IAM Group
IAM Role

Example:

Anuj
 |
 v
S3ReadPolicy

It answers:

What can this identity do?

⸻

48. Resource-Based Policy

Attached directly to an AWS resource.

Common examples:

S3 Bucket Policy
SQS Queue Policy
SNS Topic Policy
KMS Key Policy

Example:

S3 Bucket
    |
    v
Bucket Policy

It can define which principals can access that resource.

⸻

49. Identity Policy vs Resource Policy

Identity policy:

User / Role
    |
    v
What can this identity access?

Resource policy:

Resource
   |
   v
Who can access me?

Example:

IAM Role Policy
         +
S3 Bucket Policy

AWS evaluates the policies applicable to the request.

⸻

50. IAM Permissions Boundary

A permissions boundary defines:

The maximum permissions an IAM user or role can receive through identity-based policies.

Think of it as a ceiling.

Example:

Identity Policy:
S3
EC2
Glue
Lambda

Boundary:

S3
Glue

Effective permissions cannot exceed the boundary.

Concept:

Identity Policy
       ∩
Permissions Boundary
       =
Effective Permission

⸻

51. Permissions Boundary Example

Policy allows:

S3
Glue
EC2
Lambda

Boundary allows:

S3
Glue

Effective:

S3    
Glue  
EC2   
Lambda 

The boundary itself does not grant permissions.

It limits what identity-based policies can grant.

⸻

52. AWS Organizations SCP

SCP stands for:

Service Control Policy

SCPs are used in:

AWS Organizations

They define permission guardrails for accounts or organizational units.

Example organization:

Organization
   |
   +---- Development Account
   |
   +---- QA Account
   |
   +---- Production Account

Suppose the organization blocks:

ec2:TerminateInstances

Even if an IAM role inside the account has:

ec2:*

the Organizations guardrail can prevent termination when applicable.

⸻

53. IAM Policy vs Permissions Boundary vs SCP

Think:

IAM Policy
=
What identity is granted
Permissions Boundary
=
Maximum identity-based permissions
SCP
=
Organization/account-level guardrail

Simplified model:

Effective Permission
=
Allowed by relevant policies
AND
within applicable guardrails
AND
not explicitly denied

⸻

54. MFA

MFA means:

Multi-Factor Authentication

Authentication using:

Something you know
+
Something you have

Example:

Password
+
Authenticator code

MFA significantly improves account security.

Especially important for:

Root user
Privileged administrators
Sensitive operations

⸻

55. IAM Access Keys

Access keys are used for programmatic access.

They contain:

Access Key ID
Secret Access Key

Example usage:

AWS CLI
SDK
Boto3
Scripts

Avoid:

ACCESS_KEY = "ABC..."
SECRET_KEY = "XYZ..."

Hardcoding credentials is a security risk.

Prefer role-based temporary credentials whenever possible.

⸻

56. IAM Role vs Access Keys

Bad:

Application
     |
     | Hardcoded credentials
     v
AWS

Better:

Application
     |
     v
IAM Role
     |
     v
Temporary Credentials
     |
     v
AWS

Benefits:

No stored long-term secrets
Automatic temporary credential handling
Reduced credential leakage risk
Easier credential lifecycle management

⸻

57. Root User

The AWS account root user is the identity created when the AWS account is created.

Root user has extremely powerful access.

Best practices:

Do not use root for daily activities
Enable MFA
Protect root credentials
Do not create unnecessary root access keys
Use delegated administrative identities instead

⸻

58. IAM Best Practices

Important practices:

1. Follow least privilege
2. Use temporary credentials where possible
3. Prefer IAM roles for AWS workloads
4. Avoid hardcoded credentials
5. Protect privileged identities with MFA
6. Regularly review permissions
7. Remove unused credentials
8. Restrict resources where possible
9. Use Conditions where useful
10. Use central identity/federation for workforce access where appropriate
11. Monitor IAM activity using logging/security services
12. Avoid using root for everyday work

⸻

59. Real-Time Data Engineering IAM Architecture

Pipeline:

S3 Raw
   |
   v
Glue
   |
   v
S3 Processed
   |
   v
Redshift

Create:

GlueDataPipelineRole

Trust:

Glue Service
    |
    v
Can Assume Role

Permissions:

S3:
GetObject
PutObject
ListBucket
Glue Catalog:
GetDatabase
GetTable
UpdateTable
CloudWatch:
Write Logs
Redshift:
Required connectivity/API permissions

Architecture:

                        IAM
                         |
              GlueDataPipelineRole
                         |
        --------------------------------
        |              |               |
        v              v               v
       S3        Glue Catalog      CloudWatch
        ^
        |
       Glue

⸻

60. IAM Troubleshooting — AccessDenied

A common production error:

AccessDenied

Do not immediately add:

AdministratorAccess

Instead troubleshoot systematically.

Ask:

1. Which principal is making the request?
2. Which action is failing?
3. Which resource is being accessed?
4. Does identity policy allow it?
5. Does resource policy allow/deny it?
6. Is there an explicit Deny?
7. Is a permissions boundary involved?
8. Is an SCP involved?
9. Is the ARN correct?
10. Is a Condition failing?
11. For cross-account access, are both sides configured?
12. Is the application using the role you expected?

⸻

61. Example AccessDenied Troubleshooting

Error:

AccessDenied:
not authorized to perform s3:GetObject

Check:

Principal:
GlueETLRole
Action:
s3:GetObject
Resource:
arn:aws:s3:::company-data/file.csv

Then examine:

GlueETLRole policies
S3 bucket policy
Permissions boundary
SCP
Object/bucket ARN
Encryption/KMS permissions if relevant

⸻

62. Common IAM Mistakes

Mistake 1

Using:

"Action": "*",
"Resource": "*"

for convenience.

Problem:

Excessive permissions

⸻

Mistake 2

Hardcoding AWS keys.

aws_access_key_id = "..."
aws_secret_access_key = "..."

Use roles where possible.

⸻

Mistake 3

Confusing Trust and Permission policies.

Remember:

Trust
=
Who can assume?
Permission
=
What can it do?

⸻

Mistake 4

Wrong S3 ARN.

Wrong for GetObject:

arn:aws:s3:::company-data

Correct:

arn:aws:s3:::company-data/*

⸻

Mistake 5

Adding Allow without checking an explicit Deny.

Remember:

Explicit Deny wins.

⸻

63. Complete IAM Mental Model

Whenever solving an IAM issue, think:

WHO?
 |
 v
Principal
WHAT?
 |
 v
Action
ON WHAT?
 |
 v
Resource
WHO TRUSTS WHOM?
 |
 v
Trust Policy
WHAT PERMISSIONS EXIST?
 |
 v
Identity / Resource Policies
ANY GUARDRAILS?
 |
 +---- Permissions Boundary
 |
 +---- SCP
 |
 +---- Session Policy
ANY EXPLICIT DENY?
 |
 v
Final Decision

⸻

AWS IAM INTERVIEW QUESTIONS

⸻

64. What is AWS IAM?

Answer

AWS IAM stands for Identity and Access Management. It controls authentication and authorization within AWS by defining who can access AWS resources, which actions they can perform, on which resources, and under what conditions.

⸻

65. What are the main IAM components?

Answer

The major components are:

IAM Users
IAM Groups
IAM Roles
IAM Policies
Temporary credentials using AWS STS

⸻

66. What is an IAM User?

Answer

An IAM user is a long-term AWS identity created for a person or application requiring AWS access. It can have credentials and permissions assigned directly or through groups.

For modern workloads, roles and temporary credentials are generally preferred over long-lived IAM user credentials.

⸻

67. What is an IAM Group?

Answer

An IAM group is a collection of IAM users used to manage common permissions centrally.

Example:

DataEngineers
 |
 +-- Anuj
 +-- Rahul
 +-- Priya

⸻

68. Can a Group Contain Another Group?

Answer

No.

IAM groups can contain:

Users

They cannot contain:

Other Groups
IAM Roles

⸻

69. Can One User Belong to Multiple Groups?

Answer

Yes.

Example:

Anuj
 |
 +---- DataEngineers
 |
 +---- ProductionSupport

The user’s applicable permissions can come from both group memberships.

⸻

70. What is an IAM Policy?

Answer

An IAM policy is a JSON document defining permissions.

Important elements include:

Effect
Action
Resource
Condition

and depending on policy type:

Principal

⸻

71. What is Effect?

Answer

Effect specifies whether a policy statement:

Allows
or
Denies

an action.

Possible values:

Allow
Deny

⸻

72. What is Action?

Answer

Action defines which AWS API operation can be performed.

Example:

s3:GetObject
s3:PutObject
ec2:StartInstances
glue:StartJobRun

⸻

73. What is Resource?

Answer

Resource defines which AWS resource the permission applies to.

Example:

arn:aws:s3:::company-data/*

⸻

74. What is ARN?

Answer

ARN stands for:

Amazon Resource Name

It uniquely identifies AWS resources.

⸻

75. What is an IAM Role?

Answer

An IAM role is an AWS identity containing permissions that can be assumed temporarily by trusted users, AWS services, applications, or other AWS accounts.

Roles typically use temporary security credentials.

⸻

76. Why Use IAM Roles?

Answer

IAM roles avoid the need to store long-term AWS credentials.

Benefits include:

Temporary credentials
Reduced credential leakage risk
Automatic credential handling
Cross-account access
Secure AWS service-to-service access

⸻

77. What is a Trust Policy?

Answer

A trust policy determines:

Who can assume an IAM role?

Example:

Glue service
        |
        v
GlueETLRole

⸻

78. What is a Permission Policy?

Answer

A permission policy determines:

What actions the principal or role is allowed to perform.

Example:

GlueETLRole
       |
       v
s3:GetObject
s3:PutObject

⸻

79. Trust Policy vs Permission Policy

Answer

Trust Policy
=
Who can assume the role?
Permission Policy
=
What can the role do after assumption?

This is one of the most important IAM interview concepts.

⸻

80. What is AWS STS?

Answer

AWS Security Token Service provides temporary security credentials.

It is commonly used when:

Assuming roles
Cross-account access
Federated access
Temporary sessions

⸻

81. What is AssumeRole?

Answer

sts:AssumeRole allows a trusted principal to temporarily assume an IAM role and obtain temporary security credentials associated with that role.

⸻

82. What Credentials Does STS Return?

Typically:

Access Key ID
Secret Access Key
Session Token
Expiration

⸻

83. IAM User vs IAM Role

IAM User	IAM Role
Long-term identity	Assumable identity
May have password	No ordinary IAM-user password
May have long-term access keys	Temporary credentials
Often individual identity	Common for services/workloads
Credentials persist until changed	Session credentials expire

⸻

84. What is Least Privilege?

Answer

Least privilege means granting only the minimum permissions necessary to perform a required task.

Instead of:

s3:*

use:

s3:GetObject
s3:PutObject

if those are the only required actions.

⸻

85. What Happens if Both Allow and Deny Exist?

Answer

An applicable explicit:

Deny

wins over:

Allow

Remember:

Explicit Deny > Allow

⸻

86. What is Implicit Deny?

Answer

AWS denies requests by default unless an applicable permission allows them.

Example:

Policy allows:

s3:GetObject

but not:

s3:DeleteObject

The delete operation is implicitly denied unless another applicable policy grants it.

⸻

87. What is Explicit Deny?

Answer

An explicit deny occurs when a policy specifically contains:

"Effect": "Deny"

An applicable explicit Deny overrides applicable Allows.

⸻

88. What is a Resource-Based Policy?

Answer

A resource-based policy is attached directly to an AWS resource and controls which principals can access that resource.

Examples:

S3 Bucket Policy
SQS Queue Policy
SNS Topic Policy

⸻

89. Identity-Based vs Resource-Based Policy

Answer

Identity-based policies are attached to:

Users
Groups
Roles

Resource-based policies are attached to:

Resources

Think:

Identity Policy:
What can I access?
Resource Policy:
Who can access me?

⸻

90. What is a Permissions Boundary?

Answer

A permissions boundary defines the maximum permissions that identity-based policies can effectively grant to an IAM user or role.

It does not independently grant permissions.

⸻

91. What is an SCP?

Answer

An SCP is a Service Control Policy used with AWS Organizations.

It provides organization-level permission guardrails for accounts or organizational units.

SCPs do not work like ordinary identity policies that directly grant permissions.

⸻

92. What is MFA?

Answer

MFA stands for Multi-Factor Authentication.

It requires more than one authentication factor.

Example:

Password
+
Authenticator Code

⸻

93. Why Should You Avoid Hardcoding Credentials?

Answer

Hardcoded credentials can:

Leak through source code
Be committed to Git
Be copied accidentally
Become difficult to rotate
Remain active longer than intended

Prefer IAM roles and temporary credentials.

⸻

94. Scenario: EC2 Needs S3 Access

Question

Your application running on EC2 needs to read data from S3. How would you configure IAM?

Answer

Create an IAM role with the required S3 permissions and associate it with the EC2 instance through the appropriate EC2 role mechanism.

Architecture:

EC2
 |
 v
IAM Role
 |
 v
Temporary Credentials
 |
 v
S3

Do not hardcode AWS access keys inside the application.

⸻

95. Scenario: Glue Gets AccessDenied

Question

Your Glue job fails with:

AccessDenied:
s3:GetObject

How would you troubleshoot?

Answer

I would identify:

1. Which IAM role Glue is using
2. Whether role policy allows s3:GetObject
3. Whether the object ARN is correct
4. Whether S3 bucket policy allows or denies access
5. Whether an explicit Deny exists
6. Whether permissions boundary restricts it
7. Whether an Organizations policy restricts it
8. Whether encryption introduces additional permissions such as KMS access

⸻

96. Scenario: Development Needs Production S3

Question

A user in a development AWS account needs temporary access to production S3. What would you do?

Answer

Use cross-account IAM roles.

Development User
       |
       | AssumeRole
       v
ProductionReadRole
       |
       v
Production S3

Configure:

Trust Policy
+
Permission Policy

instead of creating duplicate long-term IAM credentials in the production account.

⸻

97. Scenario: User Has S3FullAccess but Delete Fails

Question

A user has an Allow for all S3 actions but still cannot delete an object. Why?

Answer

I would check for an applicable:

Explicit Deny

which might exist in:

Identity policy
Bucket policy
Permissions-related guardrail
SCP
Other applicable policy

An explicit Deny overrides an Allow.

⸻

98. Scenario: Trust Policy Allows Glue but Glue Cannot Read S3

Answer

Trusting Glue only allows Glue to assume the role.

It does not automatically grant S3 permissions.

You also need a permission policy containing something like:

s3:GetObject

Therefore:

Trust Policy
=
Can Glue assume the role?
Permission Policy
=
Can the role access S3?

⸻

99. Scenario: Permission Policy Allows S3 but Glue Cannot Assume Role

Answer

Check the role’s trust policy.

The role may have S3 permissions, but if:

glue.amazonaws.com

is not trusted appropriately, Glue cannot assume the role.

⸻

100. IAM Interview Rapid Revision

Remember these lines:

IAM
=
Authentication + Authorization
User
=
Long-term identity
Group
=
Collection of IAM users
Policy
=
JSON permission document
Role
=
Assumable identity
Trust Policy
=
Who can assume?
Permission Policy
=
What can it do?
STS
=
Temporary credentials
Least Privilege
=
Minimum required permissions
Explicit Deny

Allow
No Allow
=
Implicit Deny
Identity Policy
=
What can this principal access?
Resource Policy
=
Who can access this resource?
Permissions Boundary
=
Maximum identity-based permission
SCP
=
AWS Organizations permission guardrail

⸻

101. Final IAM Architecture to Remember

                       AWS IAM
                          |
        --------------------------------------
        |              |          |           |
      USERS          GROUPS     ROLES      POLICIES
        |              |          |           |
        |              |          |           |
   Human Identity   Organize    Temporary   Permissions
        |           Users       Identity      |
        |                        |             |
        -------------------------+-------------
                                 |
                                 v
                      AWS Policy Evaluation
                                 |
              ---------------------------------
              |                               |
           ALLOW                          DENY
                                               |
                                               v
                                  Explicit Deny Wins

⸻

102. Recommended IAM Learning Sequence

Study IAM in this order:

1. Authentication vs Authorization
2. IAM User
3. IAM Group
4. IAM Policy
5. Policy JSON Structure
6. AWS Managed Policy
7. Customer Managed Policy
8. Inline Policy
9. IAM Role
10. Trust Policy
11. Permission Policy
12. STS
13. AssumeRole
14. Temporary Credentials
15. Identity-Based Policy
16. Resource-Based Policy
17. IAM Evaluation Logic
18. Explicit vs Implicit Deny
19. Permissions Boundary
20. SCP
21. Cross-Account Access
22. MFA
23. Access Keys
24. Least Privilege
25. IAM Troubleshooting
26. Real Project IAM Architecture

⸻

103. Final Interview Answer — Explain IAM End to End

If an interviewer asks:

Explain AWS IAM.

You can answer:

AWS IAM stands for Identity and Access Management. It is used to control authentication and authorization for AWS resources.

IAM includes users, groups, roles, and policies. Users represent identities, groups help organize users with common permissions, policies are JSON documents defining what actions are allowed or denied, and roles are assumable identities commonly used by AWS services, applications, and for cross-account access.

An IAM role contains a trust relationship that determines who can assume the role and permission policies that determine what the role can do.

Roles generally use temporary security credentials provided through AWS STS, which helps avoid storing permanent access keys.

AWS evaluates applicable policies whenever a request is made. Requests are denied by default unless appropriately allowed, and an applicable explicit Deny overrides an Allow.

In production, I follow the principle of least privilege, use IAM roles instead of hardcoded credentials, protect privileged access with MFA, restrict policies to required actions and resources, and troubleshoot AccessDenied errors by checking identity policies, resource policies, trust policies, permissions boundaries, organization-level guardrails, conditions, and resource ARNs.